# FactFlow Demo Notebook

This notebook demonstrates the FactFlow agent system for sentiment-reality analysis.

## What is FactFlow?

FactFlow is a multi-agent system that compares news sentiment with actual market price action to identify market inefficiencies. It protects users from "Fake News" dumps and "Hollow Hype" pumps by providing data-driven market analysis.

### Architecture

FactFlow uses a **Sequential Multi-Agent System** with three specialized agents:

1. **News Scout Agent** - Analyzes news sentiment using Google Search
2. **Market Analyst Agent** - Fetches real-time market data (price, volume, market cap)
3. **Judge Agent** - Synthesizes findings and provides recommendations

```
User Query → News Scout → Market Analyst → Judge → Final Recommendation
```

### Key Features

- **Divergence Detection**: Identifies when sentiment and price action diverge
- **Real-Time Data**: Integration with live market APIs
- **Session Management**: Multi-turn conversations with context
- **Memory Bank**: Long-term storage of historical analyses
- **Observability**: Logging, tracing, and metrics collection


In [2]:
# Install required dependencies if not already installed
import subprocess
import sys
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')

# Map of package names to import names (for packages with different import names)
package_import_map = {
    "google-adk": "google.adk",
    "google-generativeai": "google.generativeai",
    "yfinance": "yfinance",
    "python-dotenv": "dotenv",
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy",
}

required_packages = [
    "google-adk>=0.1.0",
    "google-generativeai>=0.8.0",
    "yfinance>=0.2.0",
    "requests>=2.31.0",
    "python-dotenv>=1.0.0",
    "pandas>=2.0.0",
    "numpy>=1.24.0",
]

def check_and_install_package(package):
    """Check if a package is installed, install if not."""
    package_name = package.split(">=")[0].split("==")[0]
    import_name = package_import_map.get(package_name, package_name.replace("-", "_"))
    
    try:
        __import__(import_name)
        print(f"✅ {package_name} is already installed")
        return True
    except ImportError:
        print(f"📦 Installing {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package], 
                                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            # Verify installation
            try:
                __import__(import_name)
                print(f"✅ {package_name} installed successfully")
                return True
            except ImportError:
                print(f"⚠️  {package_name} installed but import failed")
                return False
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to install {package_name}")
            return False

print("Checking dependencies...")
print("=" * 60)
missing_packages = []
for package in required_packages:
    package_name = package.split(">=")[0].split("==")[0]
    if not check_and_install_package(package):
        missing_packages.append(package_name)

print("=" * 60)
if missing_packages:
    print(f"\n⚠️  Warning: Some packages could not be installed: {', '.join(missing_packages)}")
    print("   Please install them manually using:")
    print(f"   pip install {' '.join(missing_packages)}")
    print("\n   Or install all requirements:")
    print("   pip install -r factflow/requirements.txt")
else:
    print("\n✅ All dependencies are installed!")


Checking dependencies...
✅ google-adk is already installed
✅ google-generativeai is already installed
✅ yfinance is already installed
✅ requests is already installed
✅ python-dotenv is already installed
✅ pandas is already installed
✅ numpy is already installed

✅ All dependencies are installed!


## Setup

First, let's set up the environment and import necessary modules.


In [5]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Add the directory containing the factflow package to Python path
current_dir = Path.cwd()

# If we're in the notebooks directory, we need to go up to the parent that contains factflow
if current_dir.name == "notebooks":
    # We're in factflow/notebooks/, so factflow/ is the parent
    factflow_dir = current_dir.parent
    # The directory containing factflow/ needs to be in the path
    parent_of_factflow = factflow_dir.parent
    if str(parent_of_factflow) not in sys.path:
        sys.path.insert(0, str(parent_of_factflow))
elif current_dir.name == "factflow":
    # We're in factflow/, so add the parent directory
    parent_dir = current_dir.parent
    if str(parent_dir) not in sys.path:
        sys.path.insert(0, str(parent_dir))
else:
    # Try to find factflow directory
    factflow_path = current_dir / "factflow"
    if factflow_path.exists():
        if str(current_dir) not in sys.path:
            sys.path.insert(0, str(current_dir))
    else:
        # Look for factflow in parent directories
        for parent in current_dir.parents:
            factflow_check = parent / "factflow"
            if factflow_check.exists() and factflow_check.is_dir():
                if str(parent) not in sys.path:
                    sys.path.insert(0, str(parent))
                break

# Load environment variables - try multiple locations
env_paths = [
    current_dir.parent / ".env" if current_dir.name == "notebooks" else current_dir / ".env",  # factflow/.env
    current_dir.parent.parent / ".env" if current_dir.name == "notebooks" else current_dir.parent / ".env",  # parent/.env
    Path.home() / ".env",  # home directory
]
for env_path in env_paths:
    if env_path.exists():
        load_dotenv(dotenv_path=env_path)
        print(f"Loaded .env file.")
        break
else:
    # Try loading from current directory
    load_dotenv()

# Check for API key
if not os.getenv("GEMINI_API_KEY"):
    print("⚠️  Warning: GEMINI_API_KEY not found in environment variables.")
    print("   Please set it in your .env file or as an environment variable.")
    print("   For Kaggle notebooks, add it to Kaggle Secrets.")
else:
    print("✅ GEMINI_API_KEY found")

# Verify factflow can be imported
try:
    import factflow
    print("✅ factflow package found.")
except ImportError as e:
    print(f"❌ Error importing factflow: {e}")
    print(f"Python path: {sys.path}")
    raise

# Import FactFlow components
from factflow.session.session_manager import FactFlowSessionManager
from factflow.session.memory_service import MemoryBank
from factflow.observability.logging_config import setup_logging, get_logger
from factflow.observability.tracing import TraceCollector
from factflow.observability.metrics import MetricsCollector
from factflow.tools.market_tools import get_crypto_price_data, get_stock_data, get_tradfi_context

print("✅ All imports successful!")


Loaded .env file.
✅ GEMINI_API_KEY found
✅ factflow package found.
✅ All imports successful!


## 1. Basic Agent Usage

Let's start with a simple example of using the FactFlow agent to analyze an asset.


In [13]:
# Create a session manager
session_manager = FactFlowSessionManager()

# Get a runner for a session
runner = session_manager.get_runner(session_id="demo_session")

# Run a query
print("🔍 Analyzing Ethereum...")
print("=" * 80)

query = "Assess Ethereum right now"
response = await runner.run_debug(query)

# Extract the response text
response_text = ""
if isinstance(response, list) and len(response) > 0:
    # Find the Judge Agent's output (final recommendation)
    for item in reversed(response):
        if hasattr(item, 'content') and item.content:
            if hasattr(item.content, 'parts') and item.content.parts:
                item_text = ""
                for part in item.content.parts:
                    if hasattr(part, 'text') and part.text:
                        item_text += part.text + "\n"
                
                if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                    response_text = item_text
                    break
                elif not response_text and item_text.strip():
                    response_text = item_text

print(response_text)


🔍 Analyzing Ethereum...

 ### Created new session: debug_session_id

User > Assess Ethereum right now
NewsScoutAgent > - Sentiment Score: -4/10
- Key Themes: Market downturn, oversold conditions, bearish momentum, institutional caution, potential stabilization.
- Reasoning: Recent news indicates a significant downturn for Ethereum, with sharp price drops and underperformance compared to the broader crypto market. Technical indicators like RSI and MACD suggest oversold conditions and bearish momentum, leading to "Strong Sell" recommendations from some analysts. While there are hints of potential stabilization and a possible relief rally due to extreme fear, the overall sentiment remains negative, with warnings about increased volatility and the need for caution. Institutional interest, such as BlackRock's involvement, is mentioned but overshadowed by the current market sell-off. The Fear and Greed Index is in the "Fear" zone.


MarketAnalystAgent > Given the provided context, here's an assessment of Ethereum:

**Current Price:** $2838.33 USD
**24h Price Change:** +3.39%

**Volume Status:** Volume data is not directly comparable to "average" without historical data, but the 24h volume was $19,787,901,723.93.

**Market Cap:** $342,000,661,815.92 USD

**Market Context:** Traditional markets (S&P 500 and NASDAQ) experienced significant downturns (-2.35% and -3.58% respectively). The correlation indicator is "positive," suggesting that Ethereum's movements are currently aligned with broader traditional market trends.

**Price Action Interpretation:** Ethereum has shown a moderate positive price movement in the last 24 hours, with a 3.39% increase. This contradicts the negative sentiment from the news, which mentioned sharp price drops.

**Volume Analysis:** Without historical data, it's difficult to definitively state if the volume is spiking or low, but it represents substantial trading activity.

**Market Correl

## 2. Multiple Asset Analysis

Let's analyze multiple assets in sequence to see how the agent handles different queries.


In [14]:
# Analyze multiple assets
assets = [
    "Bitcoin",
    "Solana",
    "AAPL stock"
]

results = {}

for asset in assets:
    print(f"\n{'='*80}")
    print(f"📊 Analyzing: {asset}")
    print('='*80)
    
    query = f"Assess {asset} right now"
    response = await runner.run_debug(query)
    
    # Extract response
    response_text = ""
    if isinstance(response, list) and len(response) > 0:
        for item in reversed(response):
            if hasattr(item, 'content') and item.content:
                if hasattr(item.content, 'parts') and item.content.parts:
                    item_text = ""
                    for part in item.content.parts:
                        if hasattr(part, 'text') and part.text:
                            item_text += part.text + "\n"
                    
                    if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                        response_text = item_text
                        break
                    elif not response_text and item_text.strip():
                        response_text = item_text
    
    results[asset] = response_text
    print(response_text)
    
    # Small delay to avoid rate limiting
    await asyncio.sleep(2)

print(f"\n✅ Analyzed {len(results)} assets")



📊 Analyzing: Bitcoin

 ### Continue session: debug_session_id

User > Assess Bitcoin right now
NewsScoutAgent > - Sentiment Score: -7/10
- Key Themes: Sharp price decline, market-wide sell-off, institutional outflows, extreme fear, bearish momentum, potential bottoming.
- Reasoning: The news indicates a significant downturn for Bitcoin, with sharp price drops and a decline to seven-month lows. This is reflected in the plunge of the Crypto Fear & Greed Index to "extreme fear," its lowest level since late 2022. Institutional investors are showing caution, with record outflows from US-listed Bitcoin ETFs. This widespread sell-off, liquidations, and bearish sentiment contribute to the strongly negative sentiment score. However, some analysts suggest that the current price action could signal a potential bottoming, with some even predicting a bounce back or a healthier upward move after the deleveraging. El Salvador's continued accumulation of Bitcoin despite the market downturn is also no

MarketAnalystAgent > ## Bitcoin Market Analysis

**Current Price:** $87,564 USD
**24h Price Change:** +3.87%

**Volume Status:** The 24h volume was $57,295,354,770.08, indicating substantial trading activity. Without historical data, it's difficult to definitively classify it as above average, average, or below average.

**Market Cap:** $1,747,672,307,671.49 USD

**Market Context:** Traditional markets (S&P 500 and NASDAQ) experienced significant downturns (-2.35% and -3.58% respectively). The correlation indicator is "positive," suggesting that Bitcoin's movements are currently aligned with broader traditional market trends.

**Price Action Interpretation:** Bitcoin has shown a moderate positive price movement in the last 24 hours, with a 3.87% increase. This is a notable gain, especially considering the extremely negative sentiment described in the news context.

**Volume Analysis:** The high trading volume suggests significant market participation.

**Market Correlation:** Bitcoin a

MarketAnalystAgent > ## Solana Market Analysis

**Current Price:** $132.57 USD
**24h Price Change:** +4.90%

**Volume Status:** The 24h volume was $3,913,193,120.74. While significant, without historical context, it's hard to classify as above average, average, or below average. The news did mention a drop in trading volume.

**Market Cap:** $74,119,137,820.05 USD

**Market Context:** Traditional markets (S&P 500 and NASDAQ) experienced significant downturns (-2.35% and -3.58% respectively). The correlation indicator is "positive," suggesting that Solana's movements are currently aligned with broader traditional market trends.

**Price Action Interpretation:** Solana has shown a strong positive price movement in the last 24 hours, with a 4.90% increase. This is a significant gain, especially when contrasted with the bearish technicals and reported price declines mentioned in the news context.

**Volume Analysis:** The news indicated a drop in trading volume for Solana, which is notewor

MarketAnalystAgent > ## AAPL Stock Analysis

**Current Price:** $271.50 USD
**24h Price Change:** -0.88%

**Volume Status:** The trading volume for AAPL was 4,300,075. Without historical data, it's difficult to classify this as above average, average, or below average.

**Market Cap:** $4,029,017,227,264 USD

**Market Context:** Traditional markets (S&P 500 and NASDAQ) experienced significant downturns (-2.35% and -3.58% respectively). The correlation indicator is "positive," suggesting that AAPL's movements are generally aligned with broader market trends.

**Price Action Interpretation:** AAPL has experienced a moderate negative price movement in the last 24 hours, with a 0.88% decrease. This is a slight pullback, which contrasts with the generally positive sentiment surrounding its financial performance and analyst ratings.

**Volume Analysis:** It is difficult to assess volume status without historical data.

**Market Correlation:** AAPL's slight negative movement aligns with the p

## 3. Memory Bank Usage

The Memory Bank stores historical analyses and allows the agent to learn from past patterns.


In [7]:
# Initialize Memory Bank
memory = MemoryBank(storage_path="demo_memory.json")

# Store some sample analyses
print("💾 Storing analyses in Memory Bank...")

memory.store_analysis(
    asset="ethereum",
    sentiment_score=-8.0,
    price_change=-0.5,
    recommendation="HOLD",
    divergence_type="bullish",
)

memory.store_analysis(
    asset="bitcoin",
    sentiment_score=7.5,
    price_change=2.3,
    recommendation="BUY",
    divergence_type=None,
)

memory.store_analysis(
    asset="solana",
    sentiment_score=9.0,
    price_change=-1.2,
    recommendation="SELL",
    divergence_type="bearish",
)

print("✅ Analyses stored")


💾 Storing analyses in Memory Bank...
✅ Analyses stored


In [8]:
# Retrieve asset history
print("\n📜 Asset History:")
print("=" * 80)

ethereum_history = memory.get_asset_history("ethereum", limit=5)
print(f"\nEthereum History ({len(ethereum_history)} analyses):")
for analysis in ethereum_history:
    print(f"  - {analysis['timestamp']}: Sentiment {analysis['sentiment_score']}/10, "
          f"Price {analysis['price_change']:.2f}%, "
          f"Recommendation: {analysis['recommendation']}")

bitcoin_history = memory.get_asset_history("bitcoin", limit=5)
print(f"\nBitcoin History ({len(bitcoin_history)} analyses):")
for analysis in bitcoin_history:
    print(f"  - {analysis['timestamp']}: Sentiment {analysis['sentiment_score']}/10, "
          f"Price {analysis['price_change']:.2f}%, "
          f"Recommendation: {analysis['recommendation']}")



📜 Asset History:

Ethereum History (1 analyses):
  - 2025-11-24T00:59:53.383800: Sentiment -8.0/10, Price -0.50%, Recommendation: HOLD

Bitcoin History (1 analyses):
  - 2025-11-24T00:59:53.384798: Sentiment 7.5/10, Price 2.30%, Recommendation: BUY


In [9]:
# Get divergence patterns
print("\n🔍 Divergence Patterns:")
print("=" * 80)

patterns = memory.get_divergence_patterns(limit=10)
print(f"\nFound {len(patterns)} divergence patterns:")

for pattern in patterns:
    print(f"  - {pattern['asset']}: {pattern['divergence_type']} divergence "
          f"(Sentiment: {pattern['sentiment_score']}, Price: {pattern['price_change']:.2f}%)")



🔍 Divergence Patterns:

Found 2 divergence patterns:
  - solana: bearish divergence (Sentiment: 9.0, Price: -1.20%)
  - ethereum: bullish divergence (Sentiment: -8.0, Price: -0.50%)


In [10]:
# Get context summary
print("\n📋 Context Summary:")
print("=" * 80)

summary = memory.get_context_summary()
print(summary)

ethereum_summary = memory.get_context_summary(asset="ethereum")
print(f"\nEthereum-specific summary:\n{ethereum_summary}")



📋 Context Summary:
Memory Bank Summary: 3 total analyses, 2 divergence patterns detected.

Ethereum-specific summary:
Historical analysis for ethereum:
- 2025-11-24T00:59:53.383800: Sentiment -8.0/10, Price -0.50%, Recommendation: HOLD



## 4. Observability Features

FactFlow includes comprehensive observability features: logging, tracing, and metrics collection.


In [11]:
# Setup logging
logger = setup_logging(log_level="INFO", log_file="demo_factflow.log")
logger.info("Starting FactFlow demo with observability")

# Initialize observability components
trace_collector = TraceCollector(trace_file="demo_traces.json")
metrics_collector = MetricsCollector(metrics_file="demo_metrics.json")

print("✅ Observability components initialized")


2025-11-24 01:00:42 - factflow - INFO - Logging configured successfully
2025-11-24 01:00:42 - factflow - INFO - Starting FactFlow demo with observability
✅ Observability components initialized


In [15]:
# Run a query with full observability
query = "Assess Bitcoin right now"
session_id = "observability_demo"

# Start trace
trace_id = trace_collector.start_trace(session_id, query)
logger.info(f"Started trace: {trace_id}")

print(f"🔍 Running query with trace ID: {trace_id}")
print("=" * 80)

# Get runner
runner = session_manager.get_runner(session_id=session_id)

# Run query
response = await runner.run_debug(query)

# Extract response
response_text = ""
if isinstance(response, list) and len(response) > 0:
    for item in reversed(response):
        if hasattr(item, 'content') and item.content:
            if hasattr(item.content, 'parts') and item.content.parts:
                item_text = ""
                for part in item.content.parts:
                    if hasattr(part, 'text') and part.text:
                        item_text += part.text + "\n"
                
                if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                    response_text = item_text
                    break
                elif not response_text and item_text.strip():
                    response_text = item_text

print(response_text)

# End trace
trace = trace_collector.end_trace(final_output=response_text)
print(f"\n✅ Trace completed: {trace['trace_id']}")
print(f"   Duration: {trace.get('total_duration_ms', 0):.2f} ms")

# Record metrics
metrics_collector.record_query(success=True)
print("✅ Metrics recorded")


2025-11-24 01:06:08 - factflow - INFO - Started trace: trace_20251124_010608_995817
🔍 Running query with trace ID: trace_20251124_010608_995817

 ### Created new session: debug_session_id

User > Assess Bitcoin right now
NewsScoutAgent > Sentiment Score: -7/10

Key Themes:
*   Significant price decline and downward trend
*   Large outflows from Bitcoin ETFs
*   Increased market fear and volatility
*   Broader market weakness and liquidations

Reasoning: The recent news indicates a strong negative sentiment for Bitcoin. The price has experienced a significant drop, hitting seven-month lows and marking its worst month since 2022. There are substantial outflows from Bitcoin ETFs, signaling institutional caution. Market sentiment is described as "extreme fear," with increased volatility and widespread liquidations across the crypto market. These factors point to a prevailing bearish outlook, with concerns about potential further downside.


JudgeAgent > ## Sentiment-Reality Analysis

**Sentiment Score:** -7/10
**Price Action:** 3.94%

**Divergence Type:** Bearish Divergence

**Analysis:**
The sentiment score indicates a strongly negative outlook for Bitcoin, with themes of price decline, ETF outflows, fear, volatility, and liquidations. This is corroborated by news describing significant price drops and institutional caution. However, the Market Analyst Agent reports a positive 24-hour price change of +3.94% for Bitcoin. This creates a bearish divergence, where negative sentiment and news are not translating into immediate price weakness, and in fact, the price has moved up. This suggests that either the negative news is being ignored by the market, or there are underlying buying pressures (possibly from retail or short-covering) that are temporarily overpowering the bearish sentiment. The broader TradFi context shows weakness in the S&P 500 and Nasdaq, which typically correlates positively with Bitcoin, further adding to

In [16]:
# View trace details
print("\n📊 Trace Details:")
print("=" * 80)

if trace:
    print(f"Trace ID: {trace['trace_id']}")
    print(f"Session ID: {trace['session_id']}")
    print(f"Query: {trace['user_query']}")
    print(f"Start Time: {trace['start_time']}")
    print(f"End Time: {trace['end_time']}")
    print(f"Duration: {trace.get('total_duration_ms', 0):.2f} ms")
    print(f"\nAgents Executed: {len(trace.get('agents', []))}")
    for agent in trace.get('agents', []):
        print(f"  - {agent['agent_name']}")
        if 'tool_calls' in agent and agent['tool_calls']:
            print(f"    Tools used: {[tc['tool_name'] for tc in agent['tool_calls']]}")
    
    print(f"\nTotal Tool Calls: {len(trace.get('tool_calls', []))}")
    if trace.get('errors'):
        print(f"Errors: {len(trace['errors'])}")
    else:
        print("Errors: None ✅")



📊 Trace Details:
Trace ID: trace_20251124_010608_995817
Session ID: observability_demo
Query: Assess Bitcoin right now
Start Time: 2025-11-24T01:06:08.995817
End Time: 2025-11-24T01:06:20.570264
Duration: 11574.45 ms

Agents Executed: 0

Total Tool Calls: 0
Errors: None ✅


In [17]:
# Record some sample metrics
metrics_collector.record_sentiment_score(-8.0, asset="bitcoin")
metrics_collector.record_sentiment_score(7.5, asset="ethereum")
metrics_collector.record_divergence("bullish", -8.0, -0.5, "bitcoin")
metrics_collector.record_tool_call("get_crypto_price_data", 250.5)
metrics_collector.record_tool_call("get_stock_data", 180.2)
metrics_collector.record_agent_execution("NewsScoutAgent", 1200.0)
metrics_collector.record_agent_execution("MarketAnalystAgent", 800.0)
metrics_collector.record_agent_execution("JudgeAgent", 600.0)

# Get metrics summary
print("\n📈 Metrics Summary:")
print("=" * 80)

summary = metrics_collector.get_summary()
print(f"Total Queries: {summary['total_queries']}")
print(f"Successful Queries: {summary['successful_queries']}")
print(f"Success Rate: {summary['success_rate']:.2%}")

if 'sentiment_stats' in summary:
    print(f"\nSentiment Statistics:")
    print(f"  Count: {summary['sentiment_stats']['count']}")
    print(f"  Mean: {summary['sentiment_stats']['mean']:.2f}")
    print(f"  Min: {summary['sentiment_stats']['min']:.2f}")
    print(f"  Max: {summary['sentiment_stats']['max']:.2f}")

if 'divergence_detection_rate' in summary:
    print(f"\nDivergence Detection Rate: {summary['divergence_detection_rate']:.2%}")

if summary.get('avg_tool_latencies'):
    print(f"\nAverage Tool Latencies:")
    for tool, latency in summary['avg_tool_latencies'].items():
        print(f"  {tool}: {latency:.2f} ms")

if summary.get('avg_agent_times'):
    print(f"\nAverage Agent Execution Times:")
    for agent, time in summary['avg_agent_times'].items():
        print(f"  {agent}: {time:.2f} ms")



📈 Metrics Summary:
Total Queries: 1
Successful Queries: 1
Success Rate: 100.00%

Sentiment Statistics:
  Count: 2
  Mean: -0.25
  Min: -8.00
  Max: 7.50

Divergence Detection Rate: 100.00%

Average Tool Latencies:
  get_crypto_price_data: 250.50 ms
  get_stock_data: 180.20 ms

Average Agent Execution Times:
  NewsScoutAgent: 1200.00 ms
  MarketAnalystAgent: 800.00 ms
  JudgeAgent: 600.00 ms


## 5. Tool Testing

Let's test the individual tools that the agent uses to fetch market data.


In [18]:
# Test crypto price data tool
print("🪙 Testing Crypto Price Data Tool:")
print("=" * 80)

crypto_result = get_crypto_price_data("ethereum")
print(f"\nAsset: {crypto_result.get('asset', 'N/A')}")
if crypto_result.get('status') == 'success':
    print(f"Current Price: ${crypto_result.get('current_price', 0):,.2f}")
    print(f"24h Change: {crypto_result.get('price_change_24h', 0):.2f}%")
    print(f"24h Volume: ${crypto_result.get('volume_24h', 0):,.2f}")
    print(f"Market Cap: ${crypto_result.get('market_cap', 0):,.2f}")
else:
    print(f"Error: {crypto_result.get('error', 'Unknown error')}")


🪙 Testing Crypto Price Data Tool:

Asset: ethereum
Current Price: $2,833.10
24h Change: 3.26%
24h Volume: $19,933,419,050.08
Market Cap: $342,021,614,070.11


In [19]:
# Test stock data tool
print("\n📈 Testing Stock Data Tool:")
print("=" * 80)

stock_result = get_stock_data("AAPL")
print(f"\nSymbol: {stock_result.get('symbol', 'N/A')}")
if stock_result.get('status') == 'success':
    print(f"Current Price: ${stock_result.get('current_price', 0):,.2f}")
    print(f"24h Change: {stock_result.get('price_change_24h', 0):.2f}%")
    print(f"Volume: {stock_result.get('volume', 0):,.0f}")
    print(f"Market Cap: ${stock_result.get('market_cap', 0):,.2f}")
else:
    print(f"Error: {stock_result.get('error', 'Unknown error')}")



📈 Testing Stock Data Tool:

Symbol: AAPL
Current Price: $271.50
24h Change: -0.88%
Volume: 4,300,075
Market Cap: $4,029,017,227,264.00


In [20]:
# Test TradFi context tool
print("\n🌍 Testing TradFi Context Tool:")
print("=" * 80)

tradfi_result = get_tradfi_context()
if tradfi_result.get('status') == 'success':
    print(f"S&P 500 24h Change: {tradfi_result.get('sp500_change', 0):.2f}%")
    print(f"NASDAQ 24h Change: {tradfi_result.get('nasdaq_change', 0):.2f}%")
    print(f"Correlation Indicator: {tradfi_result.get('correlation_indicator', 'N/A')}")
else:
    print(f"Error: {tradfi_result.get('error', 'Unknown error')}")



🌍 Testing TradFi Context Tool:
S&P 500 24h Change: -2.35%
NASDAQ 24h Change: -3.58%
Correlation Indicator: positive


## 6. Session Management

FactFlow supports multi-turn conversations through session management. Let's see how the agent maintains context across multiple queries.


In [21]:
# Create a new session for multi-turn conversation
session_id = "multi_turn_demo"
runner = session_manager.get_runner(session_id=session_id)

print("💬 Multi-Turn Conversation Demo:")
print("=" * 80)

# First query
print("\n1️⃣ First Query:")
query1 = "Assess Ethereum right now"
print(f"Query: {query1}")

response1 = await runner.run_debug(query1)
response_text1 = ""
if isinstance(response1, list) and len(response1) > 0:
    for item in reversed(response1):
        if hasattr(item, 'content') and item.content:
            if hasattr(item.content, 'parts') and item.content.parts:
                item_text = ""
                for part in item.content.parts:
                    if hasattr(part, 'text') and part.text:
                        item_text += part.text + "\n"
                
                if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                    response_text1 = item_text
                    break
                elif not response_text1 and item_text.strip():
                    response_text1 = item_text

print(response_text1[:500] + "..." if len(response_text1) > 500 else response_text1)


💬 Multi-Turn Conversation Demo:

1️⃣ First Query:
Query: Assess Ethereum right now

 ### Created new session: debug_session_id

User > Assess Ethereum right now
NewsScoutAgent > - Sentiment Score: -4/10
- Key Themes: Market sell-off, oversold conditions, bearish momentum, liquidation risks, but also leading developer activity and whale accumulation.
- Reasoning: The overall sentiment for Ethereum is currently negative due to a broad crypto market sell-off, leading to a price drop and increased bearish momentum. Technical indicators show oversold conditions and significant liquidation risks. Binance has issued a "Strong Sell" recommendation for ETH/USD. However, there are some counter-balancing positive signals, such as Ethereum leading in developer activity on GitHub, which is seen as a strong indicator for long-term project strength, and reports of a large whale aggressively buying Ethereum during the downturn, signaling a potential shift in sentiment from shorting to accumulation. De

MarketAnalystAgent > The current price of Ethereum is $2833.10, with a 24-hour price change of +3.26%. The trading volume is $19,933,419,050.08, which is above average. The market cap is $342,021,614,070.11.

The traditional markets (S&P 500 and NASDAQ) experienced a significant decline in the last 24 hours, with -2.35% and -3.58% changes, respectively. Ethereum's price movement is showing a positive correlation with the traditional markets.

**Analysis:**
Despite the negative sentiment reported in the news, Ethereum has shown a moderate upward price movement in the last 24 hours, outperforming the traditional markets. The trading volume is above average, indicating strong market interest. The price action suggests a moderate upward trend, potentially driven by factors not immediately apparent in the broader market context, or a reaction to the oversold conditions and whale accumulation mentioned in the news. The positive correlation with traditional markets, however, suggests that bro

In [22]:
# Follow-up query (agent should remember context)
print("\n2️⃣ Follow-up Query:")
query2 = "What about Bitcoin?"
print(f"Query: {query2}")
print("(The agent should maintain context from the previous query)")

response2 = await runner.run_debug(query2)
response_text2 = ""
if isinstance(response2, list) and len(response2) > 0:
    for item in reversed(response2):
        if hasattr(item, 'content') and item.content:
            if hasattr(item.content, 'parts') and item.content.parts:
                item_text = ""
                for part in item.content.parts:
                    if hasattr(part, 'text') and part.text:
                        item_text += part.text + "\n"
                
                if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                    response_text2 = item_text
                    break
                elif not response_text2 and item_text.strip():
                    response_text2 = item_text

print(response_text2[:500] + "..." if len(response_text2) > 500 else response_text2)



2️⃣ Follow-up Query:
Query: What about Bitcoin?
(The agent should maintain context from the previous query)

 ### Continue session: debug_session_id

User > What about Bitcoin?
NewsScoutAgent > - Sentiment Score: -6/10
- Key Themes: Significant price drop, bearish momentum, outflows from ETFs, investor concerns, prolonged losing streak, but also positive community sentiment and short-term price recovery.
- Reasoning: Bitcoin is experiencing a predominantly negative sentiment due to a significant price drop, with reports of it falling to seven-month lows and experiencing its weakest month since 2022. There have been substantial outflows from Bitcoin ETFs, and investors are concerned about expensive tech stocks and US interest rate decisions. Analysts warn of further losses if Bitcoin drops below key support levels. The market has shown a persistent bearish bias, with Bitcoin and other major cryptocurrencies accumulating multiple losing sessions. The Crypto Fear & Greed Index is in the 

MarketAnalystAgent > The current price of Bitcoin is $87,694, with a 24-hour price change of +4.02%. The trading volume is $57,671,037,996.73, which is above average. The market cap is $1,748,334,384,871.60.

The traditional markets (S&P 500 and NASDAQ) experienced a significant decline in the last 24 hours, with -2.35% and -3.58% changes, respectively. Bitcoin's price movement is showing a positive correlation with the traditional markets.

**Analysis:**
Bitcoin has experienced a notable price increase of 4.02% in the last 24 hours, with above-average trading volume, suggesting strong market activity and buying interest. This positive price action is occurring despite a predominantly negative sentiment, significant price drops reported in news, ETF outflows, and investor concerns about the broader economic climate. The Crypto Fear & Greed Index is in the "extreme fear" zone.

Interestingly, Bitcoin's price action is showing a positive correlation with the traditional markets, which al

## Summary

This notebook demonstrated:

✅ **Basic Agent Usage** - Running queries with the FactFlow agent  
✅ **Multiple Asset Analysis** - Analyzing different assets in sequence  
✅ **Memory Bank Usage** - Storing and retrieving historical analyses  
✅ **Observability Features** - Logging, tracing, and metrics collection  
✅ **Tool Testing** - Testing individual market data tools  
✅ **Session Management** - Multi-turn conversations with context  

### Next Steps

- Try analyzing your own assets
- Experiment with different query formats
- Explore the Memory Bank patterns
- Review the trace and metrics files
- Check out the full documentation in `USAGE.md`

### Notes

- Make sure you have set your `GEMINI_API_KEY` before running
- Some cells may take 30-60 seconds to complete (API calls)
- The notebook uses async/await, so run cells in order
- For Kaggle notebooks, add your API key to Kaggle Secrets
